# Notebook 04 - Modelisation
Comparaison de quatre familles de modeles et quatre strategies de gestion du
desequilibre avec validation croisee stratifiee a cinq splits.

Pour une comparaison equitable, les modeles utilises avec SMOTE ou
undersampling ne conservent pas simultanement les poids de classes.


In [ ]:
import sys
from pathlib import Path
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier


In [ ]:
train = pd.read_csv(ROOT / 'data' / 'processed' / 'train.csv')
X_train, y_train = train.drop(columns='bad_nutrition'), train['bad_nutrition'].astype(int)
preprocessor = joblib.load(ROOT / 'models' / 'preprocessor.joblib')
class_ratio = float((y_train == 0).sum() / (y_train == 1).sum())

weighted_models = {
    'LogisticRegression': LogisticRegression(max_iter=5000, class_weight='balanced', random_state=42),
    'DecisionTree': DecisionTreeClassifier(class_weight='balanced', random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, class_weight='balanced_subsample', n_jobs=1, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, scale_pos_weight=class_ratio, n_jobs=1, random_state=42, eval_metric='logloss'),
}
unweighted_models = {
    'LogisticRegression': LogisticRegression(max_iter=5000, random_state=42),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, n_jobs=1, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, n_jobs=1, random_state=42, eval_metric='logloss'),
}
samplers = {
    'baseline': None,
    'smote': SMOTE(random_state=42),
    'undersample': RandomUnderSampler(random_state=42),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rows = []
for model_name in weighted_models:
    for strategy_name, sampler in samplers.items():
        classifier = weighted_models[model_name] if sampler is None else unweighted_models[model_name]
        steps = [('preprocessor', clone(preprocessor))]
        if sampler is not None:
            steps.append(('sampler', sampler))
        steps.append(('clf', classifier))
        pipeline = ImbPipeline(steps)
        scores = cross_val_score(pipeline, X_train, y_train, scoring='f1', cv=cv, n_jobs=1)
        rows.append({
            'model': model_name,
            'strategy': strategy_name,
            'mean_f1': scores.mean(),
            'std_f1': scores.std(),
        })
        print(f'{model_name:20s} {strategy_name:12s} F1={scores.mean():.4f} +/- {scores.std():.4f}')

results = pd.DataFrame(rows).sort_values('mean_f1', ascending=False).reset_index(drop=True)
results.to_csv(ROOT / 'models' / 'model_selection_results.csv', index=False)
assert results['model'].nunique() >= 4
assert results['strategy'].nunique() >= 3
display(results)


In [ ]:
comparison = results.assign(
    score=results.apply(lambda row: f"{row.mean_f1:.4f} +/- {row.std_f1:.4f}", axis=1)
).pivot(index='model', columns='strategy', values='score')
display(comparison)
best = results.iloc[0]
print(f"Meilleure combinaison: {best.model} + {best.strategy}, F1 CV={best.mean_f1:.4f} +/- {best.std_f1:.4f}")


## Synthese
Le meilleur couple est selectionne exclusivement sur le jeu d'entrainement via
validation croisee. Le jeu de test reste intact jusqu'a l'evaluation finale.
